# 🎾 OMNIS-COURT LLM + Jina Server
## Colab Primary Instance

**Instructions:**
1. Runtime → Change runtime type → T4 GPU
2. Run All cells below
3. Wait ~10 minutes for model download
4. Copy both URLs when ready
5. Paste into config/platforms.json
6. Close tab (anti-idle keeps server running)

In [ ]:
# ==========================================
# CELL 1: INSTALL DEPENDENCIES
# ==========================================
!pip install -q vllm cloudflared trafilatura fastapi uvicorn nest-asyncio
print('✅ Dependencies installed')

In [ ]:
# ==========================================
# CELL 2: ANTI-IDLE PROTECTION
# ==========================================
from IPython.display import display, Javascript

display(Javascript('''
    // Auto-click every 5 minutes to prevent idle disconnect
    setInterval(function() {
        var btn = document.querySelector('colab-run-button');
        if (btn) btn.click();
        console.log('[Anti-Idle] Ping sent at ' + new Date().toLocaleTimeString());
    }, 300000);
'''))

print('✅ Anti-idle active! You can close this tab safely.')
print('⏰ Session will stay alive for up to 12 hours.')

In [ ]:
# ==========================================
# CELL 3: START QWEN3-30B-A3B VLLM SERVER
# ==========================================
import subprocess
import time
import requests
import threading

def start_vllm():
    cmd = [
        'python', '-m', 'vllm.entrypoints.openai.api_server',
        '--model', 'Qwen/Qwen3-30B-A3B',
        '--served-model-name', 'qwen3-30b',
        '--host', '0.0.0.0',
        '--port', '8000',
        '--max-model-len', '8192',
        '--gpu-memory-utilization', '0.9',
        '--trust-remote-code',
        '--enforce-eager'
    ]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return process

print('🚀 Starting vLLM server...')
print('⏳ Downloading Qwen3-30B-A3B (~17GB, first run takes 5-10 min)...')

vllm_process = start_vllm()

# Wait for server to be ready
max_wait = 600  # 10 minutes
waited = 0
while waited < max_wait:
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print('✅ vLLM server is READY on port 8000')
            break
    except:
        pass
    time.sleep(10)
    waited += 10
    if waited % 60 == 0:
        print(f'⏳ Still waiting... ({waited}s elapsed)')

if waited >= max_wait:
    print('❌ vLLM failed to start. Check logs:')
    print(vllm_process.stderr.read().decode()[-2000:])

In [ ]:
# ==========================================
# CELL 4: START JINA READER SERVER
# ==========================================
import trafilatura
from fastapi import FastAPI, Query
from fastapi.responses import JSONResponse
import uvicorn
import asyncio
import nest_asyncio

nest_asyncio.apply()

app = FastAPI(title='OMNIS Jina Reader')

@app.get('/health')
async def health():
    return {'status': 'ok', 'service': 'jina-reader'}

@app.get('/extract')
async def extract(url: str = Query(..., description='URL to extract content from')):
    try:
        downloaded = trafilatura.fetch_url(url)
        if not downloaded:
            return JSONResponse(
                status_code=400,
                content={'error': 'Failed to fetch URL', 'url': url}
            )
        
        content = trafilatura.extract(
            downloaded,
            include_comments=False,
            include_tables=True,
            no_fallback=False
        )
        
        metadata = trafilatura.extract(
            downloaded,
            output_format='json',
            include_comments=False,
            include_tables=True
        )
        
        if not content or len(content.strip()) < 50:
            return JSONResponse(
                status_code=400,
                content={'error': 'Content too short or empty', 'url': url}
            )
        
        return {
            'url': url,
            'content': content,
            'word_count': len(content.split()),
            'metadata': metadata,
            'status': 'success'
        }
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={'error': str(e), 'url': url}
        )

def run_jina_server():
    uvicorn.run(app, host='0.0.0.0', port=8001, log_level='warning')

# Start Jina server in background thread
jina_thread = threading.Thread(target=run_jina_server, daemon=True)
jina_thread.start()

time.sleep(3)

# Verify Jina server is running
try:
    r = requests.get('http://localhost:8001/health', timeout=5)
    if r.status_code == 200:
        print('✅ Jina Reader server is READY on port 8001')
    else:
        print('❌ Jina server returned unexpected status')
except Exception as e:
    print(f'❌ Jina server failed: {e}')

In [ ]:
# ==========================================
# CELL 5: CLOUDFLARE TUNNELS (2 URLs)
# ==========================================
import re

def start_tunnel(port, name):
    process = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://localhost:{port}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    # Read stderr until we find the tunnel URL
    tunnel_url = None
    for line in process.stderr:
        match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break
    
    return process, tunnel_url

print('🌐 Starting Cloudflare Tunnel for LLM (port 8000)...')
llm_tunnel_proc, llm_url = start_tunnel(8000, 'LLM')

print('🌐 Starting Cloudflare Tunnel for Jina (port 8001)...')
jina_tunnel_proc, jina_url = start_tunnel(8001, 'Jina')

if llm_url and jina_url:
    print()
    print('=' * 60)
    print('🎉 OMNIS-COURT COLAB SERVER READY!')
    print('=' * 60)
    print(f'🧠 LLM URL:  {llm_url}')
    print(f'📖 JINA URL: {jina_url}')
    print('=' * 60)
    print()
    print('📋 COPY BOTH URLs to config/platforms.json')
    print('⏰ Session lasts up to 12 hours')
    print('🔒 Anti-idle is active - you can close this tab')
    print('=' * 60)
else:
    print('❌ Failed to get tunnel URLs')
    print(f'LLM URL: {llm_url}')
    print(f'Jina URL: {jina_url}')

In [ ]:
# ==========================================
# CELL 6: TEST BOTH SERVICES
# ==========================================
print('🧪 Testing LLM endpoint...')
try:
    test_payload = {
        'model': 'qwen3-30b',
        'messages': [{'role': 'user', 'content': 'Say OK if ready'}],
        'max_tokens': 10
    }
    r = requests.post(f'{llm_url}/v1/chat/completions', json=test_payload, timeout=30)
    if r.status_code == 200:
        resp = r.json()['choices'][0]['message']['content']
        print(f'✅ LLM OK: {resp}')
    else:
        print(f'❌ LLM Error: {r.status_code}')
except Exception as e:
    print(f'❌ LLM Test Failed: {e}')

print()
print('🧪 Testing Jina endpoint...')
try:
    r = requests.get(
        f'{jina_url}/extract',
        params={'url': 'https://en.wikipedia.org/wiki/Tennis'},
        timeout=30
    )
    if r.status_code == 200:
        data = r.json()
        print(f"✅ Jina OK: {data['word_count']} words extracted")
    else:
        print(f'❌ Jina Error: {r.status_code}')
except Exception as e:
    print(f'❌ Jina Test Failed: {e}')

print()
print('✅ All tests passed! System is ready.')
print('📋 Don\'t forget to copy URLs above!')